# Module 4: Fine-Tuning Results

⏱ **Estimated time:** 15 minutes

In this notebook, you'll compare a fine-tuned Nova Micro model with the untuned version on the same 50 questions. We ran the fine-tuning job ahead of time with Module 3's settings, so this notebook just reads the saved results and doesn't deploy or bill anything.

## What You'll Do

1. Compare the overall scores
2. Compare the two models question by question
3. Break the results down by area of law
4. Read one answer fine-tuning got wrong
5. Save the result for the comparison tables in Modules 5 and 6

---
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.  
SPDX-License-Identifier: MIT-0

## Configuration

Sets up the variables and AWS resources used in this notebook.

In [ ]:
# Settings, and the saved fine-tuning results from the dataset bucket
import boto3
import json
import math
import os
from collections import defaultdict
from datetime import datetime, timezone

# Settings: the AWS region, the dataset, and where the pre-run results live
REGION = os.environ.get("AWS_REGION", "us-east-1")
DOMAIN = "legal"
JURISDICTION = "au"
RESULTS_KEY = f"{DOMAIN}/finetune-prerun-results.json"   # staged with the datasets
BASELINE_MODEL = "us.amazon.nova-micro-v1:0"

def stack_outputs():
    """Every CloudFormation output in the region, by key."""
    # The CloudFormation stack records the names of the resources it created
    cf = boto3.client("cloudformation", region_name=REGION)
    return {o["OutputKey"]: o["OutputValue"]
            for page in cf.get_paginator("describe_stacks").paginate()
            for s in page["Stacks"] for o in s.get("Outputs", [])}

# Look up the dataset and results buckets
outputs = stack_outputs()
DATASET_BUCKET = outputs["DatasetBucketName"]
RESULTS_BUCKET = outputs["ResultsBucketName"]
s3 = boto3.client("s3", region_name=REGION)

# Load the results of a fine-tuning run done ahead of time, because a training job takes too long to run live.
# The file holds per-question scores for both the untuned (base) and fine-tuned Nova Micro
prerun = json.loads(s3.get_object(Bucket=DATASET_BUCKET, Key=RESULTS_KEY)["Body"].read())
ft, base = prerun["finetuned"], prerun["base"]
print(f"Loaded results for {prerun['n']} questions")
print(f"  Training:  {prerun['training_items']} items, {prerun['hyperparameters']}")
print(f"  Judge:     {prerun['judge']}")

## Step 1: Overall Scores

The score is the share of the 50 questions the judge marked correct.

The difference shows how the fine-tuned model performed compared to the untuned version of the same model.

> Your own Module 2 score for untuned Nova Micro is shown below for reference. It
> came from a separate judge run, so it may differ slightly from the untuned score
> here. The comparison uses the pre-run score because both models were scored together.

In [ ]:
# Compare the overall scores
# Each score is the share of questions the judge marked correct
print(f"  Untuned Nova Micro:     {base['score']:.2f}")
print(f"  Fine-tuned Nova Micro:  {ft['score']:.2f}")
print(f"  Difference:             {ft['score'] - base['score']:+.2f}")

# For reference: your own Module 2 score for untuned Nova Micro (newest full-size run)
yours = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=RESULTS_BUCKET, Prefix="results/cells/none-base-"):
    for obj in page.get("Contents", []):
        rec = json.loads(s3.get_object(Bucket=RESULTS_BUCKET, Key=obj["Key"])["Body"].read())
        if rec.get("n", 0) >= prerun["n"]:
            yours.append(rec)
if yours:
    mine = max(yours, key=lambda r: r["timestamp"])
    print(f"\n  Your Module 2 Nova Micro score: {mine['score']:.2f} "
          "(a separate judge run; small differences are normal)")
else:
    print("\n  No Module 2 baseline found yet. Run Module 2 to see your own score here.")

## Step 2: Question by Question

Two totals can hide a lot of movement. This counts the questions fine-tuning
**fixed** (untuned wrong, fine-tuned right) and **broke** (the reverse). The
McNemar test then checks whether that split could have occurred by chance.

> A p-value above 0.05 means the difference could be chance. In our run, fine-tuning
> fixed 6 questions and broke 5 (p = 1.00), so it made no measurable difference
> either way.

In [ ]:
# Count fixed and broken questions, and test whether the split could be chance
# Index each model's per-question results by question ID so they can be paired up
base_by_id = {str(p["id"]): p for p in base["per_item"]}
ft_by_id = {str(p["id"]): p for p in ft["per_item"]}
ids = sorted(base_by_id, key=lambda i: int(i) if i.isdigit() else i)

# Compare each question's score before and after fine-tuning
fixed = [i for i in ids if base_by_id[i]["score"] < ft_by_id[i]["score"]]
broken = [i for i in ids if base_by_id[i]["score"] > ft_by_id[i]["score"]]
same = len(ids) - len(fixed) - len(broken)

# McNemar test: a check for whether two models scored on the same questions really differ.
# It only looks at questions where the result changed. The p-value is the chance of a split
# at least this lopsided if fine-tuning made no real difference (below 0.05 is the usual cut-off)
# Exact McNemar test: under "no effect", fixed and broken are equally likely
k, n_changed = min(len(fixed), len(broken)), len(fixed) + len(broken)
p_value = min(1.0, 2 * sum(math.comb(n_changed, j) for j in range(k + 1)) / 2 ** n_changed) if n_changed else 1.0

print(f"  Fixed   {len(fixed):>3}  (untuned wrong, fine-tuned right)")
print(f"  Broke   {len(broken):>3}  (untuned right, fine-tuned wrong)")
print(f"  Same    {same:>3}")
print(f"\n  McNemar p = {p_value:.3g}: "
      f"{'unlikely to be chance' if p_value < 0.05 else 'could be chance at 50 questions'}")

## Step 3: By Category

The 50 questions cover several areas of law. View how fine-tuning did per category.

In [ ]:
# Score by category, before and after fine-tuning
# For each category, keep a running total: [question count, untuned score, fine-tuned score]
by_cat = defaultdict(lambda: [0, 0.0, 0.0])
for i in ids:
    c = by_cat[base_by_id[i]["category"]]
    c[0] += 1; c[1] += base_by_id[i]["score"]; c[2] += ft_by_id[i]["score"]

# Divide the totals by the question count to get each category's average score
print(f"  {'Category':<24} {'n':>3}  untuned → fine-tuned")
for cat, (n_cat, b, f) in sorted(by_cat.items()):
    print(f"  {cat:<24} {n_cat:>3}  {b / n_cat:.2f}    →  {f / n_cat:.2f}")

## Step 4: Inspect Fine-Tuning Answers

This prints a question untuned Nova Micro got right and the fine-tuned model got wrong, with the judge's reasoning. Change `EXAMPLE` to read another.

> Notice how the fine-tuned answer sounds confident while stating rules the law
> doesn't contain.

In [ ]:
# Show one question the fine-tuned model got wrong but the untuned model got right
EXAMPLE = 0   # index into the broken questions

# Print a model's answer, cut short after a set number of characters
def show(label, item, chars=900):
    print(f"--- {label} ({'correct' if item['score'] else 'incorrect'}) ---")
    print(item["answer"][:chars] + ("..." if len(item["answer"]) > chars else ""), "\n")

if broken:
    # Wrap around if EXAMPLE is larger than the number of broken questions
    q = broken[EXAMPLE % len(broken)]
    print(f"Question {q} ({base_by_id[q]['category']}):\n{base_by_id[q]['question']}\n")
    print(f"--- Reference answer ---\n{base_by_id[q]['reference']}\n")
    show("Untuned Nova Micro", base_by_id[q])
    show("Fine-tuned Nova Micro", ft_by_id[q])
    # Show the end of the judge's reasoning, where it gives its grade
    print(f"--- Judge on the fine-tuned answer ---\n{ft_by_id[q]['judge_explanation'][-900:]}")
else:
    print("Fine-tuning broke no questions in this run.")

## Why Fine-Tuning Didn't Help Here

The benchmark is mostly testing facts: which section of which Act says what,
and what the penalty is. Fine-tuning is a poor way to add facts to a model, for a
few reasons.

**It learns the pattern before the facts.** Fine-tuning nudges the model's
weights a small amount on each training example. With 200 items, 3 epochs and a
low learning rate, the model sees each fact only a few times. That's enough to
pick up what a good answer *looks like*: the structure, the legal phrasing, and
the habit of citing a section. It's not enough to reliably memorise which
section is the right one.

**The training and evaluation questions are different.** Module 1 kept the two
sets separate on purpose, so a higher score can't come from memorised answers.
That also means most of the facts the model trained on aren't the ones it was
tested on. To help on the benchmark, the model would need to learn the law in
general, and 200 examples isn't enough to do that.

**Training on facts the model doesn't know can make it guess more.** When the
training answers contain facts the model hasn't seen before, it learns to give a
confident, specific answer anyway. Research on this
([Gekhman et al., 2024](https://arxiv.org/abs/2405.05904)) found that fine-tuning
on unfamiliar facts increases hallucination. That fits Step 4: the fine-tuned
answer sounds sure of itself while stating rules the law doesn't contain.

**Updating the weights can disturb what the model already knew.** The weights
that change during training are also used for things outside the training set.
That's one reason fine-tuning can lose questions the model used to get right: in
our run it broke 5 while fixing 6.

**Facts in the weights can't be checked or updated.** Even if fine-tuning had
worked, the model couldn't point to its source. Every amendment to the law would
also mean training it again.

This is why Module 5 tries retrieval: it gives the model the text of the
legislation when it answers, instead of relying on what it remembers.

---

## Where Fine-Tuning Does Help

Fine-tuning works well when the gap is in **how** the model responds, not
**what** it knows:

- **Consistent output format.** Always returning JSON with fixed fields, or
  structuring an answer the way your team's templates require.
- **Tone and style.** Writing in a house style, at a set reading level, or with
  the right terminology for a profession.
- **Narrow, repeated tasks.** Classification, routing, or extracting fields from
  documents, where the labels are fixed and you have plenty of examples.
- **Smaller, cheaper models.** Training a small model on a large model's outputs
  for one task, so you get similar quality at lower cost and latency.
- **Shorter prompts.** Building long instructions or many examples into the model
  so you don't send them with every request.
- **Working with retrieval.** Training a model to use retrieved passages well:
  quoting them, citing them, and saying when they don't answer the question.
  Retrieval provides the facts, and fine-tuning shapes how they're used.

| The gap | Better fit |
|---|---|
| The model doesn't know the facts | Retrieval (Module 5) |
| The facts are there, but finding them takes several searches | Agentic search (Module 6) |
| The model knows enough, but answers in the wrong format or style | Fine-tuning |

## Step 5: Save the Result for Later Modules

Saves the fine-tuned score to `results/cells/` in S3, alongside your own results.
Modules 5 and 6 read that folder for their comparison tables, so the fine-tuned
model appears next to retrieval there.

> This should print `saved`.

In [ ]:
# Save the pre-run fine-tuned score to results/cells/ for the later comparison tables
# Build a record in the same format Module 2 used for the baseline, so the scores line up
rec = {"cell": "none", "model_label": "finetuned", "model_id": "pre-run fine-tuned Nova Micro",
       "score": ft["score"], "n": len(ft["per_item"]), "judge": prerun["judge"],
       "eval_key": prerun["eval_key"], "notebook": "Module-4-Fine-Tune-Results",
       "timestamp": datetime.now(timezone.utc).isoformat(),
       "config": {"retrieval": None, "source": "pre-run", "training_job": prerun["training_job"]},
       "per_item": [{"id": p["id"], "category": p["category"], "score": p["score"]} for p in ft["per_item"]]}
# Write it to the results bucket with a timestamp in the file name
key = f"results/cells/none-finetuned-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}.json"
s3.put_object(Bucket=RESULTS_BUCKET, Key=key, Body=json.dumps(rec).encode(), ContentType="application/json")
print(f"✓ saved s3://{RESULTS_BUCKET}/{key}")

## Key Takeaways

- Measure fine-tuning; don't assume it helps. Here the benchmark decides.
- Fine-tuning changes how a model answers. It doesn't reliably give it facts it
  never had, which is the gap this benchmark exposes.
- Fine-tuning is the right tool when the gap is format, style or a narrow task,
  not missing knowledge.
- Compare question by question, not just totals: at 50 questions, a small change
  in the total can be chance.

## Success Criteria

Before moving on, verify that you have:

- [ ] Loaded results for 50 questions
- [ ] Noted the sign of the difference between the two scores
- [ ] Seen the fixed and broken counts and the McNemar p-value
- [ ] Read one answer fine-tuning broke

## Next Steps

Open **`Module-5-RAG.ipynb`**, which tries the other lever: giving Nova Micro the
legislation at question time.